# 📓 Notebook 01 — Transformación de datos

**Sistema de Denuncias Ambientales · Uruguay · D Empathy Project**

Este notebook toma los **datos crudos abiertos** del Ministerio de Ambiente de Uruguay y los transforma en un dataset normalizado, listo para los notebooks de análisis y para alimentar el tablero.

## 🎯 Objetivo
Producir un único archivo `data/denuncias.parquet` (y su gemelo `data/denuncias.csv`) que cumpla con el esquema oficial del tablero, partiendo de dos fuentes con esquemas diferentes y un crosswalk de motivos.

## 📥 Fuentes de entrada (en `data/`)
| Archivo | Contenido | Filas | Esquema |
|---|---|---|---|
| `denuncias_ambientales2010_19.xlsx` | Denuncias 2010–2019 | 3.039 | 19 columnas (extendido) |
| `denuncias-ambientales.xlsx` | Denuncias 2023 | 1.243 | 8 columnas (simplificado) |
| `TABLAMOTIVOS.XLSX` | Crosswalk motivo → categoría | 46 motivos | matriz one-hot |
| `denuncias_ambientales_UY_DEP.xlsx` | Catálogo / Diccionario / Departamentos | — | esquema **destino** |

**Origen:** [catalogodatos.gub.uy](https://catalogodatos.gub.uy/dataset/ministerio-de-ambiente-denuncias-ambientales) — Ministerio de Ambiente, Uruguay.

## 🔧 Pasos
1. Carga de las dos fuentes crudas
2. Construcción del mapa `motivo → categoría` desde TABLAMOTIVOS
3. Normalización de departamentos (variantes ortográficas, casos inválidos)
4. Unificación al esquema target de 29 columnas
5. Geocodificación: centroide de cada departamento + jitter normal
6. Enriquecimiento temporal (año, mes, trimestre, día de semana, hora)
7. Campos del formulario nuevo (`urgencia`, `recurrencia`, etc.) quedan como `NaN` en el histórico
8. Exportación a Parquet (eficiente) y CSV (legible)

## 📤 Salida
- `data/denuncias.parquet` — dataset unificado, 4.246 filas × 29 columnas (~260 KB)
- `data/denuncias.csv` — misma data en CSV (~1 MB)

## Paso 0 — Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from unicodedata import normalize

# Configuro las rutas: ajustá DATA_DIR si tu estructura es distinta
DATA_DIR = Path("../data")
assert DATA_DIR.exists(), f"No encontré la carpeta {DATA_DIR}"

# Opciones de visualización para inspeccionar mejor
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

## Paso 1 — Inspección rápida de las fuentes crudas

Antes de transformar, miro qué tengo. Es buen hábito: si las columnas cambiaron desde la última corrida, esto lo evidencia.

In [ ]:
# Cargo el dataset 2010-2019 (esquema 'viejo' con 19 columnas)
df_old = pd.read_excel(DATA_DIR / "denuncias_ambientales2010_19.xlsx",
                       sheet_name="2010-2019")
print(f"2010-2019 → shape={df_old.shape}")
df_old.head(2)

In [ ]:
# Cargo el dataset 2023 (esquema 'simplificado' con 8 columnas)
df_new = pd.read_excel(DATA_DIR / "denuncias-ambientales.xlsx", sheet_name="DEN")
print(f"2023 → shape={df_new.shape}")
df_new.head(2)

In [ ]:
# Reviso la tabla de mapeo motivo→categoría
tm = pd.read_excel(DATA_DIR / "TABLAMOTIVOS.XLSX", sheet_name="Hoja1")
print(f"TABLAMOTIVOS → {len(tm)} motivos, {tm.shape[1]} columnas")
tm.head(5)

## Paso 2 — Constantes de referencia

Defino los 19 departamentos oficiales y sus coordenadas (capital). Las usaré para validar y para asignar lat/lon.

In [ ]:
DEPTS_OFICIALES = [
    "Artigas", "Canelones", "Cerro Largo", "Colonia", "Durazno", "Flores",
    "Florida", "Lavalleja", "Maldonado", "Montevideo", "Paysandú", "Río Negro",
    "Rivera", "Rocha", "Salto", "San José", "Soriano", "Tacuarembó", "Treinta y Tres",
]

# Centroides (latitud, longitud) de la capital de cada departamento.
# Fuente: coordenadas WGS84 aproximadas de capitales departamentales.
DEPT_COORDS = {
    "Artigas": (-30.40, -56.47), "Canelones": (-34.52, -56.28),
    "Cerro Largo": (-32.37, -54.18), "Colonia": (-34.46, -57.84),
    "Durazno": (-33.38, -56.52), "Flores": (-33.55, -56.90),
    "Florida": (-34.10, -56.21), "Lavalleja": (-34.38, -55.24),
    "Maldonado": (-34.91, -54.96), "Montevideo": (-34.90, -56.16),
    "Paysandú": (-32.32, -58.08), "Río Negro": (-33.13, -58.31),
    "Rivera": (-30.91, -55.55), "Rocha": (-34.48, -54.34),
    "Salto": (-31.39, -57.97), "San José": (-34.34, -56.71),
    "Soriano": (-33.45, -58.04), "Tacuarembó": (-31.71, -55.99),
    "Treinta y Tres": (-33.23, -54.38),
}

assert len(DEPTS_OFICIALES) == 19
assert set(DEPTS_OFICIALES) == set(DEPT_COORDS.keys())
print(f"✓ {len(DEPTS_OFICIALES)} departamentos con coordenadas.")

## Paso 3 — Normalización de departamentos

Los datos crudos tienen variantes ortográficas (`'Rio Negro'`, `'San Jose'`, `'Treinta y tres'`, `'Lavalleja '` con espacio) y valores inválidos (`'No aplica'`, `'N/A'`). Construyo un normalizador robusto.

**Decisión**: los multi-departamento (ej. `'Canelones, Florida, Lavalleja'`) y los `'No aplica'` se descartan. Son ~37 filas (0.86% del total) y meterles un valor inventado sería más malo que perderlas.

In [ ]:
def _strip_accents(s: str) -> str:
    """Quita tildes/diacríticos. 'Río' → 'Rio'"""
    return "".join(c for c in normalize("NFD", s) if c.isascii() or c.isspace())

# Construyo un diccionario de variantes → nombre oficial
DEPT_FIX = {}
for d in DEPTS_OFICIALES:
    DEPT_FIX[d] = d
    DEPT_FIX[d.lower()] = d
    DEPT_FIX[_strip_accents(d).lower()] = d   # 'rio negro' → 'Río Negro'
    DEPT_FIX[_strip_accents(d)] = d           # 'Rio Negro' → 'Río Negro'
# Casos manuales que vi en los datos
DEPT_FIX["Treinta y tres"] = "Treinta y Tres"

INVALIDOS = {"no aplica", "no especifica", "n/a", "n/a - (no aplica)"}

def normalizar_departamento(raw):
    """Limpia espacios, tildes y capitalización. Devuelve el nombre oficial o None."""
    if pd.isna(raw):
        return None
    s = str(raw).strip()
    if not s or s.lower() in INVALIDOS:
        return None
    if "," in s:        # multi-dpto: descarto
        return None
    if s in DEPT_FIX:
        return DEPT_FIX[s]
    s_norm = _strip_accents(s).lower()
    return DEPT_FIX.get(s_norm)

# Smoke test
for raw in ["Montevideo", "Rio Negro", "San Jose", "Treinta y tres",
            "Lavalleja ", "No aplica", "Canelones, Florida", None]:
    print(f"  {raw!r:35s} → {normalizar_departamento(raw)!r}")

## Paso 4 — Mapa de motivos → 9 categorías ambientales

`TABLAMOTIVOS.XLSX` tiene una matriz one-hot: cada motivo está marcado con `1` en una (o varias) de 9 columnas que representan las categorías CAT_01 a CAT_09.

**Decisión sobre multi-categoría**: 3 motivos tocan más de una categoría. Para no depender del orden alfabético de las columnas (arbitrario), defino reglas explícitas. Esto vale para el alumno: cuando hay ambigüedad, decidí *con criterio sustantivo*, no algorítmico.

In [ ]:
COL_TO_CAT = {
    "FAUNA":                        ("CAT_01", "Fauna silvestre"),
    "FAJA COSTERA":                 ("CAT_02", "Costa y faja costera"),
    "CONTAMINACION AIRE":           ("CAT_03", "Contaminación del aire"),
    "CONTAMINACION CAUCES DE AGUA": ("CAT_04", "Contaminación del agua"),
    "RESIDUOS":                     ("CAT_05", "Residuos y basura"),
    "FLORA":                        ("CAT_06", "Flora y vegetación"),
    "CONTAMINACION SONORA":         ("CAT_07", "Contaminación sonora"),
    "EXTRACCION ACTIV PRODUCTIVAS": ("CAT_08", "Extracción y act. productivas"),
    "OTROS":                        ("CAT_09", "Otro problema ambiental"),
}
COLS_DUMMY = list(COL_TO_CAT.keys())

# Reglas explícitas para motivos multi-categoría:
# - Humedales tocan Fauna+Costa+Agua+Flora → priorizo Flora (categoría más específica al ecosistema)
# - Fajas de amortiguación tocan Costa+Agua → priorizo Costa (es donde se ejerce la infracción)
OVERRIDES_MULTI = {
    "Afectación de humedales y/o áreas de interés ecosistémico": "FLORA",
    "Incumplimiento de la faja de amortiguación en la Laguna del Sauce": "FAJA COSTERA",
    "Incumplimiento de la faja de amortiguación en el Río Santa Lucía": "FAJA COSTERA",
}

def construir_mapa_motivos():
    """Devuelve dict {motivo_str: (categoria_codigo, categoria_label)}."""
    mapa = {}
    for _, row in tm.iterrows():
        motivo = str(row["motivo_1"]).strip()
        if not motivo or motivo == "nan":
            continue
        if motivo in OVERRIDES_MULTI:
            mapa[motivo] = COL_TO_CAT[OVERRIDES_MULTI[motivo]]
        else:
            activas = [c for c in COLS_DUMMY if row[c] == 1]
            mapa[motivo] = COL_TO_CAT[activas[0]] if activas else COL_TO_CAT["OTROS"]
    return mapa

mapa_motivos = construir_mapa_motivos()
print(f"✓ {len(mapa_motivos)} motivos mapeados a 9 categorías.")

# Verifico la distribución
from collections import Counter
Counter(lab for _, lab in mapa_motivos.values()).most_common()

## Paso 5 — Transformación al esquema target

Defino dos funciones, una por fuente. Cada una devuelve un DataFrame con el mismo conjunto de columnas. Los campos del formulario nuevo (`urgencia`, `recurrencia`, `tipo_denunciante`, etc.) los dejo como `NaN`: no existen en datos históricos y meterles un valor por defecto sería inventar información.

In [ ]:
def cargar_dataset_2010_2019(mapa):
    """Toma el Excel 2010-2019 y lo transforma al esquema target."""
    df = pd.read_excel(DATA_DIR / "denuncias_ambientales2010_19.xlsx",
                       sheet_name="2010-2019")
    out = pd.DataFrame()
    # ID único: prefijo URY- al num_interno original.
    # Algunos num_interno aparecen duplicados (mismo expediente con varios motivos);
    # les agrego sufijo -A, -B para mantener unicidad sin perder trazabilidad.
    base_ids = df["num_interno"].astype(str).apply(lambda x: f"URY-{x}")
    cumcount = base_ids.groupby(base_ids).cumcount()
    suffix = cumcount.map(lambda i: "" if i == 0 else f"-{chr(64+i)}")  # '', '-A', '-B'...
    out["id_denuncia"]       = base_ids + suffix
    out["timestamp"]         = pd.to_datetime(df["fecha_denuncia"])
    out["fecha_hecho"]       = pd.to_datetime(df["fecha_denuncia"])
    out["departamento"]      = df["departamento"].apply(normalizar_departamento)
    out["ciudad_localidad"]  = df["localidad"]
    out["via_ingreso"]       = df["via_ingreso"]

    motivo_clean = df["motivo_1"].astype(str).str.strip()
    cat_info = motivo_clean.map(lambda m: mapa.get(m, ("CAT_09", "Otro problema ambiental")))
    out["categoria_codigo"]  = cat_info.map(lambda t: t[0])
    out["categoria_label"]   = cat_info.map(lambda t: t[1])
    out["subcategoria"]      = motivo_clean      # el motivo original funciona como subcategoría
    out["motivo_original"]   = motivo_clean      # lo conservo aparte por trazabilidad
    out["descripcion_libre"] = df["denuncia"]
    out["fuente"]            = "historico_2010_2019"
    return out

df1 = cargar_dataset_2010_2019(mapa_motivos)
print(f"2010-2019 transformado: {df1.shape}, departamentos no resueltos: {df1['departamento'].isna().sum()}")
df1.head(3)

In [ ]:
def cargar_dataset_2023(mapa):
    """Toma el Excel 2023 y lo transforma al esquema target."""
    df = pd.read_excel(DATA_DIR / "denuncias-ambientales.xlsx", sheet_name="DEN")
    out = pd.DataFrame()
    # En este dataset no hay num_interno: genero IDs secuenciales
    out["id_denuncia"]       = [f"URY-2023-{i:04d}" for i in range(1, len(df)+1)]
    out["timestamp"]         = pd.to_datetime(df["FECHA DENUNCIA"])
    out["fecha_hecho"]       = pd.to_datetime(df["FECHA DENUNCIA"])
    out["departamento"]      = df["DEPARTAMENTO"].apply(normalizar_departamento)
    out["ciudad_localidad"]  = pd.NA
    out["via_ingreso"]       = df["Vía de ingreso"]

    motivo_clean = df["MOTIVO 1"].astype(str).str.strip()
    cat_info = motivo_clean.map(lambda m: mapa.get(m, ("CAT_09", "Otro problema ambiental")))
    out["categoria_codigo"]  = cat_info.map(lambda t: t[0])
    out["categoria_label"]   = cat_info.map(lambda t: t[1])
    out["subcategoria"]      = motivo_clean
    out["motivo_original"]   = motivo_clean
    out["descripcion_libre"] = pd.NA
    out["fuente"]            = "historico_2023"
    return out

df2 = cargar_dataset_2023(mapa_motivos)
print(f"2023 transformado: {df2.shape}, departamentos no resueltos: {df2['departamento'].isna().sum()}")
df2.head(3)

## Paso 6 — Unificación

Concateno ambas fuentes y descarto las filas sin departamento válido.

In [ ]:
df = pd.concat([df1, df2], ignore_index=True)
print(f"Total antes de filtrar:  {len(df):,} filas")
df = df.dropna(subset=["departamento"]).reset_index(drop=True)
print(f"Total después de filtrar: {len(df):,} filas")
print(f"Filas descartadas (depto inválido): {(3039+1243) - len(df)}")

## Paso 7 — Geocodificación con jitter

Los datos históricos no traen lat/lon. Asigno el centroide del departamento y le agrego ruido normal (jitter) para que los puntos no se encimen en mapas.

**Importante**: estas coordenadas **NO** indican el lugar exacto del hecho. Sirven para visualizar densidad geográfica a nivel departamental. El formulario nuevo sí captura coordenadas reales (vía URL de Google Maps), y se distinguen porque tendrán `fuente='formulario'`.

In [ ]:
def agregar_coordenadas(df, jitter_std=0.15, seed=42):
    """Asigna lat/lon = centroide del depto + ruido normal de desvío `jitter_std` grados.
    Con std=0.15° el jitter cubre aprox 15-20 km, suficiente para distribuir puntos
    dentro de cada departamento sin sugerir precisión que no tenemos."""
    rng = np.random.default_rng(seed)
    lats, lons = [], []
    for d in df["departamento"]:
        base_lat, base_lon = DEPT_COORDS[d]
        lats.append(round(base_lat + rng.normal(0, jitter_std), 4))
        lons.append(round(base_lon + rng.normal(0, jitter_std), 4))
    df = df.copy()
    df["latitud"]  = lats
    df["longitud"] = lons
    return df

df = agregar_coordenadas(df)
df[["departamento", "latitud", "longitud"]].head()

## Paso 8 — Enriquecimiento temporal

Agrego columnas calculadas a partir del timestamp. Es información redundante (se puede recuperar siempre desde `timestamp`) pero acelera mucho los análisis.

In [ ]:
def agregar_columnas_temporales(df):
    df = df.copy()
    df["año"]        = df["timestamp"].dt.year
    df["mes"]        = df["timestamp"].dt.month
    df["trimestre"]  = df["timestamp"].dt.quarter
    df["semana_año"] = df["timestamp"].dt.isocalendar().week.astype(int)
    df["dia_semana"] = df["timestamp"].dt.dayofweek   # 0=lunes
    df["hora"]       = df["timestamp"].dt.hour
    df["mes_nombre"] = df["timestamp"].dt.strftime("%b")
    return df

df = agregar_columnas_temporales(df)
df.columns.tolist()[-7:]

## Paso 9 — Campos del formulario nuevo (NaN en histórico)

El formulario ciudadano de captura tiene campos que no existían en los datos oficiales (urgencia, recurrencia, tipo de denunciante, etc.). Los agrego como `NaN` para que la unión futura entre histórico + formulario sea limpia.

In [ ]:
CAMPOS_FORMULARIO = [
    "tipo_denunciante", "recurrencia", "urgencia",
    "denuncia_previa", "organismo_previo",
    "referencia_lugar", "url_mapa", "adjunto_url",
]
for col in CAMPOS_FORMULARIO:
    df[col] = pd.NA

print(f"✓ Agregados {len(CAMPOS_FORMULARIO)} campos del formulario (vacíos en histórico).")

## Paso 10 — Orden final de columnas y validación

Aplico el orden del diccionario oficial (hoja `DICCIONARIO` del archivo `denuncias_ambientales_UY_DEP.xlsx`), agrupando las columnas calculadas al final.

In [ ]:
COLS_ORDEN = [
    # Identidad
    "id_denuncia", "timestamp", "fecha_hecho",
    # Denunciante (NaN en histórico)
    "tipo_denunciante",
    # Geografía
    "departamento", "ciudad_localidad", "referencia_lugar",
    "url_mapa", "latitud", "longitud",
    # Categorización
    "categoria_codigo", "categoria_label", "subcategoria", "motivo_original",
    # Detalle
    "descripcion_libre", "recurrencia", "urgencia",
    "denuncia_previa", "organismo_previo", "adjunto_url",
    "via_ingreso", "fuente",
    # Calculadas
    "año", "mes", "trimestre", "semana_año", "dia_semana", "hora", "mes_nombre",
]
df = df[COLS_ORDEN].sort_values("timestamp").reset_index(drop=True)

print(f"✓ Dataset final: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"✓ Período cubierto: {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")

## Paso 11 — Validaciones de integridad

Antes de exportar verifico algunas cosas que tienen que cumplirse sí o sí.

In [ ]:
# 1. id_denuncia único
n_dup = df["id_denuncia"].duplicated().sum()
assert n_dup == 0, f"Hay {n_dup} IDs duplicados!"
print(f"✓ Todos los id_denuncia son únicos ({len(df):,})")

# 2. Todos los departamentos son válidos
depts_no_validos = set(df["departamento"].unique()) - set(DEPTS_OFICIALES)
assert not depts_no_validos, f"Departamentos no válidos: {depts_no_validos}"
print(f"✓ Los {df['departamento'].nunique()} departamentos son válidos")

# 3. Todas las categorías son CAT_01..CAT_09
cats_no_validas = set(df["categoria_codigo"].unique()) - {f"CAT_{i:02d}" for i in range(1,10)}
assert not cats_no_validas, f"Categorías no válidas: {cats_no_validas}"
print(f"✓ Las {df['categoria_codigo'].nunique()} categorías son válidas")

# 4. Lat/lon dentro del rango Uruguay (con margen para el jitter del centroide)
assert df["latitud"].between(-35.6, -29.5).all(),  "Latitud fuera del rango UY"
assert df["longitud"].between(-58.8, -53).all(), "Longitud fuera del rango UY"
print(f"✓ Todas las coordenadas dentro del rango geográfico de Uruguay")

# 5. Sin timestamps en el futuro
assert (df["timestamp"] <= pd.Timestamp.now()).all(), "Hay timestamps futuros"
print(f"✓ Ningún timestamp en el futuro")

## Paso 12 — Resumen y export

Veo cómo quedaron los datos y los exporto en dos formatos:
- **Parquet**: comprimido, eficiente, conserva tipos de datos → fuente principal para el resto de notebooks y para la app
- **CSV**: legible, abrible desde Excel/Sheets → útil para inspección manual o compartir

In [ ]:
# Resumen rápido
print("📊 Distribución por categoría:")
print(df["categoria_label"].value_counts().to_string())
print(f"\n📊 Distribución por fuente:")
print(df["fuente"].value_counts().to_string())
print(f"\n📊 Distribución por año:")
print(df["año"].value_counts().sort_index().to_string())

In [ ]:
out_parquet = DATA_DIR / "denuncias.parquet"
out_csv     = DATA_DIR / "denuncias.csv"

df.to_parquet(out_parquet, index=False)
df.to_csv(out_csv, index=False)

print(f"💾 Exportado:")
print(f"   {out_parquet}  ({out_parquet.stat().st_size // 1024} KB)")
print(f"   {out_csv}      ({out_csv.stat().st_size // 1024} KB)")

---
## ✅ Listo

Ahora tenés `data/denuncias.parquet` listo para usar en los siguientes notebooks:

- **02 — Descriptivo**: temporal, categorías, geografía, perfil
- **03 — Diagnóstico**: correlaciones, anomalías, triage
- **04 — Predictivo**: forecast, NLP, alertas, clustering
- **05 — Nueva denuncia**: formulario y persistencia

Para volver a generar el dataset (por ejemplo cuando llegue una versión más actualizada del catálogo de datos abiertos), simplemente reemplazá los archivos en `data/` y volvé a correr este notebook de arriba abajo.